In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import pairwise_distances

In [2]:
PROJECT_ROOT = Path("..")

EVENT_FILE = PROJECT_ROOT / "data" / "raw" / "glof_event_ledger.csv"
LAKE_FILE = PROJECT_ROOT / "data" / "final" / "glofguard_training_data.csv"

events = pd.read_csv(EVENT_FILE)
lakes = pd.read_csv(LAKE_FILE)

print("Events:", events.shape)
print("Lakes:", lakes.shape)

print("\nEvent columns:")
print(events.columns.tolist())

print("\nLake columns:")
print(lakes.columns.tolist())

Events: (15, 8)
Lakes: (8806, 13)

Event columns:
['event_id', 'event_date', 'event_name', 'latitude', 'longitude', 'source', 'event_type', 'verification_status']

Lake columns:
['sample_id', 'area', 'longitude', 'latitude', 'is_top200', 'temperature', 'rainfall', 'label', 'elevation', 'distance_to_nearest_settlement_km', 'nearest_settlement_name', 'annual_area_change_km2_per_year', 'growth_data_available']


In [3]:
display(events)

,event_id,event_date,event_name,latitude,longitude,source,event_type,verification_status
0,PMD-1999-001,1999-08-06,Khalti/Gupis,NaN,NaN,PMD historical GLOF table,GLOF,documented
1,PMD-2000-001,2000-06-10,Shimshal,NaN,NaN,PMD historical GLOF table,GLOF,documented
2,PMD-2000-002,2000-07-27,Kand/Hushe,NaN,NaN,PMD historical GLOF table,GLOF,documented
3,PMD-2007-001,2007-04-05,Ghulkin,NaN,NaN,PMD historical GLOF table,GLOF,documented
4,PMD-2008-001,2008-01-06,Passu,NaN,NaN,PMD historical GLOF table,GLOF,documented
5,PMD-2008-002,2008-04-02,Ghulkin,NaN,NaN,PMD historical GLOF table,GLOF,documented
6,PMD-2008-003,2008-05-22,Ghulkin,NaN,NaN,PMD historical GLOF table,GLOF,documented
7,PMD-2008-004,2008-05-24,Ghulkin,NaN,NaN,PMD historical GLOF table,GLOF,documented
8,PMD-2008-005,2008-06-14/15,Ghulkin,NaN,NaN,PMD historical GLOF table,GLOF,documented
9,PMD-2009-001,2009-03-26,Ghulkin,NaN,NaN,PMD historical GLOF table,GLOF,documented


In [4]:
print(lakes[["sample_id", "latitude", "longitude"]].head())

print("\nMissing coordinates:")
print(lakes[["latitude", "longitude"]].isna().sum())

   sample_id   latitude  longitude
0          1  34.828994  74.061891
1          2  34.820079  74.086340
2          3  34.806459  74.071990
3          4  34.857497  74.076763
4          5  34.861163  74.078262

Missing coordinates:
latitude     0
longitude    0
dtype: int64


### Load HMAGLOFDB and the final lake dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree

PROJECT_ROOT = Path("..")
HMA_FILE = PROJECT_ROOT / "data" / "raw" / "HMAGLOFDB.csv"
LAKE_FILE = PROJECT_ROOT / "data" / "final" / "glofguard_training_data.csv"

hma = pd.read_csv(HMA_FILE, encoding="latin1", sep=None, engine="python")
lakes = pd.read_csv(LAKE_FILE)

print("HMAGLOFDB shape:", hma.shape)
print("Lake dataset shape:", lakes.shape)

HMAGLOFDB shape: (768, 59)
Lake dataset shape: (8806, 13)


### Filter to Pakistan GLOF events only

In [2]:
pk_events = hma[hma["Country"].astype(str).str.strip().str.lower() == "pakistan"].copy()
print("Pakistan GLOF events:", len(pk_events))
pk_events[["Year_exact", "Month", "Day", "Lat_lake", "Lon_lake", "Lake_name", "Glacier_name"]].head(20)

Pakistan GLOF events: 151


,Year_exact,Month,Day,Lat_lake,Lon_lake,Lake_name,Glacier_name
2,1661.0,NaN,NaN,35.687000,75.907000,Unnamed,Biafo
4,1833.0,NaN,NaN,36.824000,74.369000,Unknown,Yashkuk-Yaz
9,1844.0,NaN,NaN,36.621400,74.079060,Karambar,Unnamed
11,1850.0,NaN,NaN,35.232000,74.722000,Unknown,Tarshing
13,1860.0,NaN,NaN,36.621400,74.079060,Karambar,Unnamed
14,1860.0,7.0,NaN,35.668000,75.378000,Unknown,Unnamed
16,1865.0,NaN,NaN,36.794600,73.994940,Sokhter Rabot,Unnamed
17,1865.0,NaN,NaN,36.621400,74.079060,Karambar,Unnamed
20,1873.0,NaN,NaN,36.510556,74.850591,Unnamed,Batura
25,1882.0,9.0,NaN,36.344239,75.474953,Unnamed,Khurdopin


### Spatially match each Pakistan event to the nearest lake, using the same 1 km threshold used throughout this project

In [3]:
event_coords = np.radians(pk_events[["Lat_lake", "Lon_lake"]].values)
lake_coords = np.radians(lakes[["latitude", "longitude"]].values)

tree = BallTree(lake_coords, metric="haversine")
distances, indices = tree.query(event_coords, k=1)

pk_events["matched_sample_id"] = lakes.iloc[indices.flatten()]["sample_id"].values
pk_events["match_distance_km"] = distances.flatten() * 6371

matched = pk_events[pk_events["match_distance_km"] <= 1].copy()
print("Events matched within 1 km:", len(matched))
print("Unique lakes matched:", matched["matched_sample_id"].nunique())

Events matched within 1 km: 27
Unique lakes matched: 14


### Check how many matched events fall within the real weather data's coverage window (2016-present)


In [4]:
weather = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "real_daily_weather.csv")
weather_lake_ids = set(weather["lake_id"].str.extract(r"(\d+)")[0].astype(int))

matched["sample_id_int"] = matched["matched_sample_id"].astype(int)
matched["has_weather_coverage"] = matched["sample_id_int"].isin(weather_lake_ids)
matched["within_weather_period"] = matched["Year_exact"] >= 2016

usable = matched[matched["has_weather_coverage"] & matched["within_weather_period"]]

print("Matched events with a lake that has weather data:", matched["has_weather_coverage"].sum())
print("Matched events after 2016 (weather period):", matched["within_weather_period"].sum())
print("Matched events usable for a forecasting dataset (both conditions):", len(usable))
print()
print(usable[["Year_exact", "Month", "Day", "matched_sample_id", "Lake_name"]])

Matched events with a lake that has weather data: 0
Matched events after 2016 (weather period): 1
Matched events usable for a forecasting dataset (both conditions): 0

Empty DataFrame
Columns: [Year_exact, Month, Day, matched_sample_id, Lake_name]
Index: []


### External validation: compare the trained model's scores for lakes near real historical events vs other lakes

In [6]:
import joblib

# Open the saved "box" and take out what's inside
saved = joblib.load(PROJECT_ROOT / "artifacts" / "final_model.joblib")
final_model = saved["pipeline"]      # the actual trained model
feature_cols = saved["features"]     # the list of column names it expects

# Mark which lakes are near a real historical event (from Step 3's matching)
lakes["near_real_event"] = lakes["sample_id"].isin(matched["matched_sample_id"].unique())

# Ask the model to score every lake
lakes["model_score"] = final_model.predict_proba(lakes[feature_cols])[:, 1]

# Compare: does the model score "near real event" lakes higher?
print(lakes.groupby("near_real_event")["model_score"].describe())

                  count      mean       std       min       25%       50%  \
near_real_event                                                             
False            8792.0  0.286767  0.275807  0.001087  0.059596  0.180302   
True               14.0  0.656349  0.269666  0.282664  0.398341  0.664411   

                      75%       max  
near_real_event                      
False            0.434602  0.999375  
True             0.918791  0.976955  


## Save matched events as a validation artifact

In [7]:
matched.to_csv(
    PROJECT_ROOT / "artifacts" / "hmaglofdb_matched_events.csv",
    index=False
)

print("Saved: artifacts/hmaglofdb_matched_events.csv")

Saved: artifacts/hmaglofdb_matched_events.csv


In [8]:
from pathlib import Path

model_file = PROJECT_ROOT / "artifacts" / "final_model.joblib"
features_file = PROJECT_ROOT / "artifacts" / "feature_cols.joblib"

print("Final model exists:", model_file.exists())
print("Feature columns file exists:", features_file.exists())

Final model exists: True
Feature columns file exists: True


### Conclusion

The HMAGLOFDB database contains 151 documented GLOF events in Pakistan.
Using a 1 km nearest-lake spatial matching threshold, 27 events were
matched to 14 unique lakes in the GLOFGuard lake inventory.

However, none of the matched events had both a corresponding lake weather
record and an event date within the available 2016-present weather period.
Therefore, these historical events cannot be used to construct a reliable
temporal forecasting dataset from the currently available weather data.

As an external spatial validation check, the 14 lakes associated with
matched historical GLOF events received a mean model score of 0.656,
compared with 0.287 for the remaining 8,792 lakes. This shows that the
current susceptibility model assigns higher scores, on average, to lakes
associated with documented historical GLOF events.

This result is treated as external spatial validation rather than proof of
future GLOF forecasting accuracy. The current model uses proxy labels
rather than verified event-timed GLOF labels. A true temporal forecasting
model would require verified historical event dates together with
continuous environmental observations covering the periods before those
events.

Therefore, the current GLOFGuard model should be described as an
ML-based GLOF susceptibility/risk-ranking model, not as a validated
real-time GLOF prediction system.